# Unidad 2: Aprendizaje Automático 
## Regresión Logística
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![curvado](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-wal_-172619-2156618639-36550471.jpg)

[Foto de wal_ 172619](https://www.pexels.com/es-es/foto/bancos-curvos-de-madera-en-blanco-y-negro-abstracto-36550471/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/04_LogisticRegression.ipynb)
 
## Flujo completo: datos → modelo → predicción → evaluación

Este notebook muestra el pipeline completo de Machine Learning supervisado con **Regresión Logística**:

```
📂 Datos  →  🔧 Preparación  →  ✂️ Split  →  🏗️ Modelo  →  🔮 Predicción  →  📊 Evaluación
```

### 🤔 ¿Qué es la Regresión Logística?

A pesar de su nombre, es un algoritmo de **clasificación** (no de regresión). Modela la probabilidad de que una muestra pertenezca a una clase usando la función sigmoide:

$$P(y=1|x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$

Si $P > 0.5$ → clase 1 (sobrevivió). Si $P \leq 0.5$ → clase 0 (no sobrevivió). 🚢

## 📦 1. Importación de librerías

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

## 📂 2. Carga y preparación de los datos

El modelo necesita entradas **numéricas**. La columna `Sex` es categórica (texto), por eso la convertimos a una variable binaria `male` (True/False → 1/0).

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/data/ml/titanic.csv')
print(type(df))
print("🔍 Dataset original (primeras filas):")
print(df.head())

In [ ]:
# Convertimos 'Sex' en feature binaria
df['male'] = df['Sex'] == 'male'  # True si es hombre, False si es mujer

print("✅ Dataset con feature 'male' agregada:")
print(df.head())

## ✂️ 3. Definición de X e y, y separación de datos

**Variables independientes (X):** Las características que el modelo usará para predecir.

**Variable objetivo (y):** Lo que queremos predecir (`Survived`).

In [ ]:
# Variables predictoras
X = df[['Pclass', 'male', 'Age', 'Siblings/Spouses', 'Parents/Children', 'Fare']].values
print(type(X))
# Variable objetivo
y = df['Survived'].values
print(type(y))
# División: 70% entrenamiento, 30% testeo (random_state fija la semilla para reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=54
)

print(f"📐 Estructura de X completo: {X.shape}")
print(f"🏋️  Estructura de X_train:   {X_train.shape}")
print(f"🧪  Estructura de X_test:    {X_test.shape}")

## 🏗️ 4. Armado del modelo y ajuste (fit)

`.fit()` entrena el modelo: ajusta los pesos $w$ e intercepto $b$ minimizando la función de pérdida logarítmica (log-loss) sobre los datos de entrenamiento.

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)
print(type(model))
print("✅ Modelo entrenado")
print(f"\n🔢 Coeficientes (pesos w): {model.coef_}")
print(f"⚓ Intercepto (bias b):    {model.intercept_}")

## 🔮 5. Predicciones sobre el conjunto de testeo

`.predict()` aplica la función sigmoide con los pesos aprendidos y devuelve la **clase predicha** (0 o 1).

In [ ]:
print("🔮 Primeras 20 predicciones:", model.predict(X_test[:20]))
print("🎯 Primeros 20 reales:      ", y_test[:20])

## 🧪 6. Predicción sobre datos nuevos

Podemos usar el modelo para predecir sobre **cualquier pasajero** que definamos manualmente.

Formato: `[Pclass, male, Age, Siblings/Spouses, Parents/Children, Fare]`

In [ ]:
# Ejemplo: Jack, pasajero de 3ra clase, hombre, 22 años, con 1 hermano, sin hijos, pagó 7.25
pasajero_nuevo = [[3, True, 22.0, 1, 0, 7.25]]

prediccion = model.predict(pasajero_nuevo)
probabilidad = model.predict_proba(pasajero_nuevo)

estado = '✅ SOBREVIVIÓ' if prediccion[0] == 1 else '❌ NO SOBREVIVIÓ'
print(f"Pasajero: {pasajero_nuevo[0]}")
print(f"Predicción: {estado}")
print(f"Prob. no sobrevivir: {probabilidad[0][0]:.4f}")
print(f"Prob. sobrevivir:    {probabilidad[0][1]:.4f}")

## 📊 7. Evaluación del modelo

Calculamos la **accuracy** (proporción de predicciones correctas sobre el total) de tres formas equivalentes.

In [ ]:
y_pred = model.predict(X_test)

aciertos = (y_test == y_pred).sum()
total = len(y_pred)

print(f"\n✔️  Total de aciertos: {aciertos} de {total}")
print(f"📐 Promedio de aciertos (manual):    {aciertos / y_test.shape[0]:.8f}")
print(f"🎯 Score / Accuracy del modelo:      {model.score(X_test, y_test):.8f}")

## 📉 8. Curva de probabilidades predichas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

y_proba = model.predict_proba(X_test)[:, 1]  # Probabilidad de sobrevivir

# Ordenar por probabilidad para una visualización más clara
idx = np.argsort(y_proba)

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(range(len(y_proba)), y_proba[idx], c=y_test[idx],
           cmap='RdYlBu', edgecolors='gray', s=30, alpha=0.8, linewidths=0.5)
ax.axhline(0.5, color='black', linestyle='--', label='Umbral = 0.5')
ax.set_xlabel('Muestras (ordenadas por probabilidad)')
ax.set_ylabel('P(Survived = 1)')
ax.set_title('Probabilidades predichas — Titanic (azul = sobrevivió, rojo = no sobrevivió)')
ax.legend()
plt.tight_layout()
plt.show()

## 🏁 9. Conclusiones

- 🚢 El flujo de ML supervisado sigue siempre el mismo esquema: **datos → preparación → split → modelo → predicción → evaluación**.
- 🔑 La feature de mayor impacto suele ser `male` (género), reflejando que las mujeres tenían mayor prioridad de evacuación.
- 🎛️ La función `predict_proba()` nos da la probabilidad de cada clase, no solo la decisión binaria — útil para ajustar el umbral de clasificación.
- 🔢 El `random_state` en `train_test_split` garantiza que el experimento sea **reproducible**.
- 📈 Para profundizar: podés explorar cambiar el umbral de decisión (por defecto 0.5) para optimizar Recall o Precision según el caso de uso.

---

© 2026 Cátedra Inteligencia Artificial — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).